In [1]:
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
import os
import random

In [2]:
# --- FUNCIONES DE PROCESAMIENTO (MISMAS DEL PASO ANTERIOR) ---

def generate_white_noise(length):
    return np.random.normal(0, 1, length)

def generate_pink_noise(length):
    white = generate_white_noise(length)
    white_fft = np.fft.rfft(white)
    frequencies = np.maximum(np.arange(len(white_fft)), 1)
    pink_filter = 1.0 / np.sqrt(frequencies)
    pink_fft = white_fft * pink_filter
    pink = np.fft.irfft(pink_fft, n=length)
    return pink

def inject_noise(signal, noise, snr_db):
    power_signal = np.mean(signal ** 2)
    power_noise = np.mean(noise ** 2)
    
    snr_linear = 10 ** (snr_db / 10.0)
    desired_power_noise = power_signal / snr_linear
    scale_factor = np.sqrt(desired_power_noise / power_noise)
    
    augmented_signal = signal + scale_factor * noise
    
    if np.max(np.abs(augmented_signal)) > 1.0:
        augmented_signal = augmented_signal / np.max(np.abs(augmented_signal))
        
    return augmented_signal

# --- PROCESAMIENTO EN LOTE (BATCH PROCESSING) ---

def pipeline_aumentacion_lote(carpeta_origen, carpeta_destino):
    # Crear las rutas usando Path (funciona en Windows, Mac y Linux)
    ruta_origen = Path(carpeta_origen)
    print (ruta_origen);
    ruta_destino = Path(carpeta_destino)
    
    # Crear la carpeta de destino si no existe
    ruta_destino.mkdir(parents=True, exist_ok=True)
    
    # Buscar todos los archivos .wav en la carpeta
    archivos_wav = list(ruta_origen.glob('*.wav'))
    kick_signals = [librosa.load(p)[0] for p in Path().glob('audio_impacto_original\*.wav')]
    print("Cantidad de audios cargados: ",len(kick_signals))  #longitud del vector, depende de la cantidad de audios
    print("Tipo de dato en cada registro: ",kick_signals[0].dtype)
    #en este caso el sistema retorna el tipo de datos del audio, en este caso dtype('float32')
    #_____________________inicio codigo para guardar nombre audios__________________________

    #if not archivos_wav:
    #    print(f"No se encontraron archivos .wav en {carpeta_origen}")
    #    return

    #print(f"Se encontraron {len(archivos_wav)} archivos. Iniciando aumentación...")

    #_____________________inicio codigo para guardar nombre audios__________________________
    ejemplo_dir='audio_impacto_original/';
    contenido = os.listdir(ejemplo_dir);
    nombre_audio_wav = []
    for fichero in contenido:
        if os.path.isfile(os.path.join(ejemplo_dir, fichero)) and fichero.endswith('.wav'):
            nombre_audio_wav.append(fichero)
    print('Nombres de los archivos de audio a procesar:');
    print(nombre_audio_wav)
    #________________________fin de cargar nombres de los registros____________________________

    #_________inicio de cargar los archivos de fondo___________________
    y1, sr1 = librosa.load("ruido_microfono/1_birds_wind[1].wav", sr=None)
    num_muestras1 = len(y1)
    y2, sr2 = librosa.load("ruido_microfono/2_mic_noise[1].wav", sr=None)
    num_muestras2 = len(y2)
    y3, sr3 = librosa.load("ruido_microfono/3_wind_blowing[1].wav", sr=None)#viento que sopla
    num_muestras3 = len(y3)
    #viento en la hierba alta
    y4, sr4 = librosa.load("ruido_microfono/4_wind_in_tall_grass[1].wav", sr=None)#viento que soplacant_sonidos=len(kick_signals)
    num_muestras4 = len(y4)
    y5, sr5 = librosa.load("ruido_microfono/5_bird[1].wav", sr=None)#cuervo pájaro chillón
    num_muestras5 = len(y5)
    y6, sr6 = librosa.load("ruido_microfono/6_mic_distortion[1].wav", sr=None)#cuervo pájaro chillón
    num_muestras6 = len(y6)
    y7, sr7 = librosa.load("ruido_microfono/7_distant_thunder[1].wav", sr=None)#rayo lejano
    num_muestras7 = len(y7)
    y8, sr8 = librosa.load("ruido_microfono/8_nearby_thunder[1].wav", sr=None)#rayo cercano
    num_muestras7 = len(y8)

    cant_sonidos=len(nombre_audio_wav);
    for i in range(cant_sonidos):
    #for i in range(1):    
        print('______________________i= '+str(i)+' ___________________');
        print("Tipo de dato en cada registro: ",kick_signals[i].dtype)
        nombre_arch=nombre_audio_wav[i];
        audio="audio_impacto_original"+ chr(92)+nombre_arch;
        
        print('Nombre del archivo a procesar:',audio);

        #snr_db = random.randint(12, 18); # valor intermedio
        
        #SNR de 5 dB a 10 dB (Ruido Alto / Estresante): El río se escucha sepultado por una tormenta o viento fuerte.
        #Obliga al modelo a aprender patrones profundos y evita el sobreajuste (overfitting).
        snr_db = random.randint(5, 10); # Ruido Alto / Estresante
        #-------crear vectores de audios de fondo-------
        snr_text=str(snr_db);
        
    #for ruta_audio in archivos_wav:    
        try:
            # 1. Cargar el audio
            y, sr = librosa.load(audio, sr=None)
            num_muestras = len(y)
            
            # 2. Generar los ruidos
            ruido_blanco = generate_white_noise(num_muestras)
            ruido_rosa = generate_pink_noise(num_muestras)
            
            # 3. Aplicar la aumentación
            audio_blanco = inject_noise(y, ruido_blanco, snr_db)
            audio_rosa = inject_noise(y, ruido_rosa, snr_db)

            # 4 construir el archivo que suma cantos de pájaros
            recorte=y1[0:num_muestras].copy();
            audio_bird=inject_noise(y, recorte, snr_db)
            recorte=y2[0:num_muestras].copy();
            audio_noise_mic=inject_noise(y, recorte, snr_db)
            recorte=y3[0:num_muestras].copy();
            audio_wind_blowing=inject_noise(y, recorte, snr_db)
            recorte=y4[0:num_muestras].copy();
            audio_wind_grass=inject_noise(y, recorte, snr_db)
            recorte=y5[0:num_muestras].copy();
            audio_crow_bird=inject_noise(y, recorte, snr_db)
            recorte=y6[0:num_muestras].copy();
            audio_distortion_mic=inject_noise(y, recorte, snr_db)
            recorte=y7[0:num_muestras].copy();
            audio_distant_thunder=inject_noise(y, recorte, snr_db)
            recorte=y8[0:num_muestras].copy();
            audio_nearby_thunder=inject_noise(y, recorte, snr_db)

            
            # 4. Definir nombres de salida manteniendo el nombre original
            #nombre_base = ruta_audio.stem  # Nombre sin el ".wav"
            
            nombre_base = nombre_arch.removesuffix(".wav")  # Nombre sin el ".wav"
            print(nombre_base)
            
            ruta_salida_blanco = ruta_destino / f"{nombre_base}_white_{snr_text}.wav"
            ruta_salida_rosa = ruta_destino / f"{nombre_base}_pink_{snr_text}.wav"
            ruta_salida_bird= ruta_destino / f"{nombre_base}_bird_{snr_text}.wav"
            ruta_salida_noise_mic= ruta_destino / f"{nombre_base}_noise_mic_{snr_text}.wav"
            ruta_salida_wind_blowing= ruta_destino / f"{nombre_base}_wind_blowing_{snr_text}.wav"
            ruta_salida_wind_grass= ruta_destino / f"{nombre_base}_wind_grass_{snr_text}.wav"
            ruta_salida_crow_bird= ruta_destino / f"{nombre_base}_crow_bird_{snr_text}.wav"
            ruta_salida_distortion_mic= ruta_destino / f"{nombre_base}_distort_mic_{snr_text}.wav"
            ruta_salida_distant_thunder= ruta_destino / f"{nombre_base}_distant_thunder_{snr_text}.wav"
            ruta_salida_nearby_thunder= ruta_destino / f"{nombre_base}_nearby_thunder_{snr_text}.wav"
            
            # 5. Guardar los archivos
            sf.write(ruta_salida_blanco, audio_blanco, sr)
            sf.write(ruta_salida_rosa, audio_rosa, sr)
            sf.write(ruta_salida_bird, audio_bird, sr)
            sf.write(ruta_salida_noise_mic, audio_noise_mic, sr)
            sf.write(ruta_salida_wind_blowing, audio_wind_blowing, sr)
            sf.write(ruta_salida_wind_grass, audio_wind_grass, sr)
            sf.write(ruta_salida_crow_bird, audio_crow_bird, sr)
            sf.write(ruta_salida_distortion_mic, audio_distortion_mic, sr)
            sf.write(ruta_salida_distant_thunder, audio_distant_thunder, sr)
            sf.write(ruta_salida_nearby_thunder, audio_nearby_thunder, sr)
            
            #print(f"Procesado con éxito: {ruta_audio.name}")
            
        except Exception as e:
            print(f"Error procesando el archivo {ruta_audio.name}: {e}")

    print("\n¡Proceso por lote terminado con éxito!")

<h1>Estrategia recomendada</h1>
<p>
Para que tu modelo de Machine Learning sea robusto, no uses un solo valor. Genera variantes utilizando un rango aleatorio de SNR:</p>
<ul>
<li>SNR de 20 dB a 25 dB (Ruido Bajo): El río se escucha con total claridad. Útil para que el modelo aprenda las características puras del agua con perturbaciones mínimas de los micrófonos.</li>
<li>SNR de 12 dB a 18 dB (Ruido Medio): Es el escenario real más común. El ruido ambiental (viento suave, hojas) compite moderadamente con el sonido del río.</li>
<li>SNR de 5 dB a 10 dB (Ruido Alto / Estresante): El río se escucha sepultado por una tormenta o viento fuerte. Obliga al modelo a aprender patrones profundos y evita el sobreajuste (overfitting).</li>
</ul>


In [3]:
# --- EJECUCIÓN ---
# Reemplaza estas rutas con las de tu computadora (pueden ser carpetas relativas o absolutas)
# C:\Usuarios\olger_erazo_dlc\
CARPETA_CON_MIS_AUDIOS = "C:/Usuarios/olger_erazo_dlc/udistrital_2026/audio_impacto_original/"
CARPETA_PARA_GUARDAR = "audio_impacto_aumentado"
SNR_CONFIG = 5  # Nivel de ruido en decibelios

pipeline_aumentacion_lote(CARPETA_CON_MIS_AUDIOS, CARPETA_PARA_GUARDAR)

C:\Usuarios\olger_erazo_dlc\udistrital_2026\audio_impacto_original
Cantidad de audios cargados:  54
Tipo de dato en cada registro:  float32
Nombres de los archivos de audio a procesar:
['20221204_154417[1].wav', '20221204_154537[1].wav', 'p_B1F_video_230726_piedras_siavenida_mono.wav', 'p_B1G_video_230726_piedras_siavenida_mono.wav', 'p_B2G_video_230726_piedras_siavenida_mono.wav', 'p_B3FM_video_240326_163546_siavenida_3.wav', 'p_B3H_video_230726_piedras_siavenida_mono.wav', 'p_B4H_video_230902_piedras_siavenida_1_mono.wav', 'p_B5F_video_230902_piedras_siavenida_1_mono.wav', 'p_B5H_video_230726_piedras_siavenida_mono.wav', 'p_represa_rio_encano_240616_si_avenida_131855_suena_piedras[13].wav', 'p_represa_rio_encano_240616_si_avenida_131855_suena_piedras[21].wav', 'p_represa_rio_encano_240616_si_avenida_131855_suena_piedras[8].wav', 'p_represa_rio_encano_240616_si_avenida_135234_sonido_piedras[14].wav', 'p_represa_rio_encano_240616_si_avenida_140009_sonido_piedras[1].wav', 'p_represa_rio